# CEdit: smallest-cosine residual subspace trên 100 celebrities (Vast AI)

Notebook này chạy riêng `smallest_cosine_subspace` trên bài toán xóa 100 celebrities, sinh ảnh `erase`/`retain`, rồi chạy GIPHY Celebrity Detector (GCD) từ CE-Eval. Notebook dành cho PyTorch Jupyter image trên Vast AI; không dùng Google Colab hoặc legacy.

> Trước khi chạy: chọn instance có GPU, mở Jupyter từ Vast AI và bảo đảm `/workspace` là persistent volume. Một lượt benchmark đầy đủ có **500 erase + 500 retain ảnh**. Để chỉ chạy một split trước, đặt `CONTENTS = ("erase",)` ở cell cấu hình.

In [ ]:
# Kiểm tra GPU trong Vast AI PyTorch Jupyter
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi tiếp tục."
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

## 1. Chuẩn bị codebase trong persistent workspace

Mặc định notebook dùng `/workspace/CEdit` và `/workspace/CE-Eval`. Có thể đổi root bằng biến môi trường `VAST_WORKSPACE`; branch CEdit được chọn bởi `CEDIT_BRANCH`. Branch đó phải chứa `smallest_cosine_subspace`.

In [ ]:
%%bash
set -Eeuo pipefail
VAST_WORKSPACE_DIR="${VAST_WORKSPACE:-/workspace}"
cd "${VAST_WORKSPACE_DIR}"

CEDIT_BRANCH="${CEDIT_BRANCH:-main}"

if [[ ! -d CEdit/.git ]]; then
  git clone --branch "${CEDIT_BRANCH}" --single-branch https://github.com/izahai/CEdit.git CEdit
else
  git -C CEdit fetch origin "${CEDIT_BRANCH}:refs/remotes/origin/${CEDIT_BRANCH}"
fi
if [[ ! -d CE-Eval/.git ]]; then
  git clone https://github.com/izahai/CE-Eval.git
fi

git -C CEdit checkout "${CEDIT_BRANCH}" 2>/dev/null || \
  git -C CEdit checkout --track -b "${CEDIT_BRANCH}" "origin/${CEDIT_BRANCH}"
git -C CEdit merge --ff-only "origin/${CEDIT_BRANCH}"
git -C CE-Eval fetch origin
git -C CE-Eval checkout 5322a63

printf 'CEdit:  '; git -C CEdit branch --show-current; printf ' @ '; git -C CEdit rev-parse --short HEAD
printf 'CE-Eval: '; git -C CE-Eval rev-parse --short HEAD

In [ ]:
# Giữ PyTorch/CUDA đã có trong Vast AI image; không cài lại torch.
%pip install -q -r /workspace/CEdit/requirements.txt
%pip install -q python-dotenv openpyxl scikit-image scikit-learn opencv-python-headless tensorflow-cpu

# TensorFlow được CE-Eval dùng cho MTCNN face detector.
import tensorflow as tf
print("TensorFlow:", tf.__version__)

Nếu Hugging Face yêu cầu token để tải Stable Diffusion v1.4, đặt biến môi trường `HF_TOKEN` trước khi mở Jupyter hoặc export trong terminal Vast AI. Cell dưới đây không in token ra output.

In [ ]:
try:
    import os
    from huggingface_hub import login
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print("Đã đăng nhập Hugging Face.")
    else:
        print("Không có HF_TOKEN; tiếp tục bằng truy cập công khai/cache.")
except Exception as exc:
    print("Bỏ qua đăng nhập Hugging Face:", type(exc).__name__)

## 2. Cấu hình thí nghiệm

- `smallest_cosine_subspace`: chọn `RESIDUAL_TOP_K` residual có mean cosine nhỏ nhất, dựng residual subspace, rồi dùng hướng ngược projection target với norm residual gốc.
- `FORCE_RETRAIN/FORCE_RESAMPLE = False` giúp chạy lại notebook mà không lặp công việc đã hoàn tất.
- Mặc định mọi artifact được ghi vào persistent volume `/workspace`.

In [ ]:
from pathlib import Path
import os, subprocess, pandas as pd

WORKSPACE = Path(os.environ.get("VAST_WORKSPACE", "/workspace"))
CEDIT = WORKSPACE / "CEdit"
CEEVAL = WORKSPACE / "CE-Eval"
SD_CKPT = "CompVis/stable-diffusion-v1-4"

BENCHMARK_NAME = "100_celebrity"
ANCHOR_CONCEPTS = "person"
ANCHOR_MODES = ("smallest_cosine_subspace",)
RESIDUAL_TOP_K = 10  # chỉnh trong khoảng [1, 100]
RESIDUAL_SCALE = 1.0
CONTENTS = ("erase", "retain")  # smoke test nhanh: ("erase",)
BATCH_SIZE = 10
FORCE_RETRAIN = False
FORCE_RESAMPLE = False
GCD_USE_CUDA = True  # CPU an toàn hơn: tránh TensorFlow và PyTorch tranh VRAM

benchmark = pd.read_csv(CEDIT / f"data/{BENCHMARK_NAME}.csv")
TARGET_CONCEPTS = ", ".join(
    benchmark.loc[benchmark["type"] == "erase", "concept"]
    .drop_duplicates()
    .tolist()
)
expected_counts = benchmark.groupby("type").size().to_dict()

assert len(TARGET_CONCEPTS.split(", ")) == 100, "Benchmark phải có đúng 100 target celebrities"
assert 1 <= RESIDUAL_TOP_K <= 100, RESIDUAL_TOP_K
assert expected_counts == {"erase": 500, "retain": 500}, expected_counts
assert "smallest_cosine_subspace" in (CEDIT / "train_erase_null.py").read_text(), (
    "Branch CEdit hiện tại chưa chứa mode smallest_cosine_subspace"
)

# Output riêng theo top-k, tránh dùng nhầm checkpoint giữa các ablation.
OUTPUT_ROOT = Path(
    WORKSPACE / f"cedit_ce_eval_outputs_{BENCHMARK_NAME}_smallest_cosine_top_k_{RESIDUAL_TOP_K}"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Số target celebrities:", len(TARGET_CONCEPTS.split(", ")))
print("Residual top-k:", RESIDUAL_TOP_K)
print("Số ảnh benchmark mỗi model:", expected_counts)
print("Output:", OUTPUT_ROOT)


Trên Vast AI, giữ `/workspace` gắn với persistent volume. Có thể đổi `OUTPUT_ROOT` ở cell tiếp theo sang một thư mục khác trong volume trước khi train.

In [ ]:
# Tùy chọn: đổi vị trí output trong persistent volume Vast AI
# OUTPUT_ROOT = Path('/workspace/my_experiments/CEdit-CE-Eval-100-smallest-cosine')
# OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 3. Train checkpoint smallest-cosine subspace

Giữ config ablation: retain scale 0.05, tắt filter, `aug_num=0`, retain set `data/100_celebrity.csv` và cột `concept`. Method nhận `--residual_top_k`; đây là ablation cô lập residual, không phải SPEED end-to-end `aug_num=10`.

In [ ]:
import os
import subprocess
import sys

def run_checked(args, cwd=CEDIT, env_extra=None):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTHONUNBUFFERED"] = "1"
    if env_extra:
        env.update({k: str(v) for k, v in env_extra.items()})

    command = [str(x) for x in args]
    print("\n" + "=" * 80, flush=True)
    print("$", " ".join(command), flush=True)
    print("=" * 80, flush=True)

    # Sử dụng Popen để đọc log theo thời gian thực
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # In từng dòng ngay khi nhận được
    for line in process.stdout:
        print(line, end="", flush=True)

    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, command)

checkpoint_paths = {}

_ANCHOR_MODES_ITERABLE = (
    [ANCHOR_MODES]
    if isinstance(ANCHOR_MODES, str)
    else ANCHOR_MODES
)

for anchor_mode in _ANCHOR_MODES_ITERABLE:
    save_dir = OUTPUT_ROOT / "checkpoints" / anchor_mode
    ckpt = save_dir / "weight.pt"
    checkpoint_paths[anchor_mode] = ckpt

    if ckpt.exists() and not FORCE_RETRAIN:
        print(f"[SKIP] Đã tồn tại checkpoint cho {anchor_mode}: {ckpt}", flush=True)
        continue

    save_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n[TRAIN] Bắt đầu train anchor mode: {anchor_mode}", flush=True)

    run_checked([
        "python",
        "-u",
        "train_erase_null.py",
        "--sd_ckpt", SD_CKPT,
        "--target_concepts", TARGET_CONCEPTS,
        "--anchor_concepts", ANCHOR_CONCEPTS,
        "--anchor_mode", anchor_mode,
        "--retain_scale", "0.05",
        "--disable_filter",
        "--aug_num", "0",
        "--retain_path", f"data/{BENCHMARK_NAME}.csv",
        "--heads", "concept",
        "--save_path", save_dir,
        "--file_name", "weight",
        "--residual_scale", RESIDUAL_SCALE,
        "--residual_top_k", RESIDUAL_TOP_K,
    ])

    assert ckpt.is_file(), f"Không tìm thấy checkpoint: {ckpt}"
    print(f"[DONE] Hoàn thành {anchor_mode}", flush=True)

checkpoint_paths

## 4. Infer ảnh `erase` và `retain`

Mỗi mode đi vào một thư mục độc lập. Cấu trúc cuối cùng là `images/<mode>/100_celebrity/<erase|retain>/edit/*.png`, đúng yêu cầu CE-Eval rằng folder truyền vào chỉ chứa file ảnh, không chứa thư mục con.

In [ ]:
def image_folder(model_name, content, sample_mode="edit"):
    return OUTPUT_ROOT / "images" / model_name / BENCHMARK_NAME / content / sample_mode

def png_count(folder):
    return len(list(folder.glob("*.png"))) if folder.exists() else 0

for anchor_mode, ckpt in checkpoint_paths.items():
    assert ckpt.is_file(), f"Thiếu checkpoint: {ckpt}"
    for content in CONTENTS:
        out = image_folder(anchor_mode, content)
        expected = expected_counts[content]
        if png_count(out) == expected and not FORCE_RESAMPLE:
            print(f"Skip infer {anchor_mode}/{content}: đủ {expected} ảnh")
            continue
        run_checked([
            "python", "sample2.py",
            "--sd_ckpt", SD_CKPT,
            "--erase_type", BENCHMARK_NAME,
            "--target_concept", BENCHMARK_NAME,
            "--contents", content,
            "--mode", "edit",
            "--batch_size", str(BATCH_SIZE),
            "--save_root", OUTPUT_ROOT / "images" / anchor_mode,
            "--edit_ckpt", ckpt,
        ])
        actual = png_count(out)
        assert actual == expected, f"{out}: cần {expected}, hiện có {actual}"

for mode in ANCHOR_MODES:
    print(mode, {c: png_count(image_folder(mode, c)) for c in CONTENTS})

In [ ]:
!ls -R

### Xem nhanh một vài ảnh

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

VIEW_SEED = 7
rng = random.Random(VIEW_SEED)

VIEW_METHODS = [(mode, "edit") for mode in ANCHOR_MODES]

# Chọn một filename chung cho từng split, để tất cả các hàng dùng cùng prompt/seed.
selected_files = {}
for content in CONTENTS:
    files_by_method = {
        label: {p.name: p for p in image_folder(label, content, sample_mode).glob("*.png")}
        for label, sample_mode in VIEW_METHODS
    }
    common_names = set.intersection(*(set(files) for files in files_by_method.values()))
    selected_files[content] = rng.choice(sorted(common_names)) if common_names else None

fig, axes = plt.subplots(
    len(VIEW_METHODS), len(CONTENTS),
    figsize=(4 * len(CONTENTS), 4 * len(VIEW_METHODS)),
)
axes = np.array(axes).reshape(len(VIEW_METHODS), len(CONTENTS))

for row, (label, sample_mode) in enumerate(VIEW_METHODS):
    for col, content in enumerate(CONTENTS):
        filename = selected_files[content]
        ax = axes[row, col]
        if filename is None:
            ax.set_title(f"{label} / {content}\nNo shared sample")
            ax.axis("off")
            continue
        image_path = image_folder(label, content, sample_mode) / filename
        ax.imshow(Image.open(image_path))
        ax.set_title(f"{label} / {content}\n{filename}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Define the root directory for images
image_root = OUTPUT_ROOT / "images" / "smallest_cosine_subspace" / BENCHMARK_NAME / "erase"

# Find all PNG files recursively in the directory
all_image_files = list(image_root.glob('**/*.png'))

if not all_image_files:
    print(f"No PNG images found in {image_root}")
else:
    # Select a maximum of 5 random images to display
    num_images_to_display = min(5, len(all_image_files))
    selected_images = random.sample(all_image_files, num_images_to_display)

    print(f"Displaying {len(selected_images)} random images from {image_root}:")

    plt.figure(figsize=(15, 5))
    for i, img_path in enumerate(selected_images):
        plt.subplot(1, num_images_to_display, i + 1)
        try:
            img = Image.open(img_path)
            plt.imshow(img)
            plt.title(img_path.name, fontsize=8)
            plt.axis('off')
        except Exception as e:
            plt.title(f"Error loading {img_path.name}\n{e}", fontsize=8)
            plt.axis('off')
    plt.tight_layout()
    plt.show()

## 5. Chuẩn bị GIPHY Celebrity Detector

CE-Eval đã chứa detector và script tải `resources.tar.gz`. Không cài bộ requirements năm 2018 của upstream (`torch 0.4`, TensorFlow 1.15, NumPy 1.15) vì chúng không tương thích image PyTorch Jupyter hiện tại; notebook dùng các package hiện đại và hai compatibility patch nhỏ không làm đổi metric.

In [ ]:
# PyTorch >=2.6 mặc định weights_only=True; checkpoint GIPHY cũ cần trusted full load.
resnet_file = CEEVAL / "celeb-detection-oss/model_training/helpers/resnet_model.py"
text = resnet_file.read_text()
text = text.replace(
    "checkpoint = torch.load(self.weights_path)",
    "checkpoint = torch.load(self.weights_path, weights_only=False)",
)
text = text.replace(
    "checkpoint = torch.load(self.weights_path, map_location=lambda storage, loc: storage)",
    "checkpoint = torch.load(self.weights_path, map_location=lambda storage, loc: storage, weights_only=False)",
)
text = text.replace("nn.Softmax()(fc2_output)", "nn.Softmax(dim=1)(fc2_output)")
resnet_file.write_text(text)
print("Đã áp compatibility patch:", resnet_file)

In [ ]:
# Tải MTCNN weights, recognition weights và labels vào examples/resources.
subprocess.run(
    ["bash", str(CEEVAL / "run/download_resources_colab.sh"), str(CEEVAL / "celeb-detection-oss")],
    check=True,
)
resources = CEEVAL / "celeb-detection-oss/examples/resources"
required = [
    resources / "face_detection/det1.npy",
    resources / "face_detection/det2.npy",
    resources / "face_detection/det3.npy",
    resources / "face_recognition/labels.csv",
    resources / "face_recognition/best_model_states.pkl",
]
missing = [str(p) for p in required if not p.is_file()]
assert not missing, "Thiếu GCD resources: " + repr(missing)
print("GCD resources OK:", resources)

## 6. Chạy CE-Eval cho từng mode và từng split

Script GCD đọc celebrity kỳ vọng từ tên file mà `sample2.py` tạo. Vì vậy phải truyền chính folder `.../<content>/edit`, không truyền thư mục cha. Log chi tiết được ghi ra file để notebook không bị hàng nghìn dòng dự đoán.

In [ ]:
gcd_output = OUTPUT_ROOT / "gcd"
gcd_output.mkdir(parents=True, exist_ok=True)
eval_targets = [(m, c, "edit") for m in _ANCHOR_MODES_ITERABLE for c in CONTENTS]

for model_name, content, sample_mode in eval_targets:
    folder = image_folder(model_name, content, sample_mode)
    xlsx = gcd_output / f"{model_name}_{content}.xlsx"
    csv = gcd_output / f"{model_name}_{content}.csv"
    log = gcd_output / f"{model_name}_{content}.log"
    if csv.is_file() and not FORCE_RESAMPLE:
        print("Skip GCD, đã có:", csv)
        continue
    env = os.environ.copy()
    env.update({
        "CELEB_DIR": str(CEEVAL / "celeb-detection-oss"),
        "EVALUATE_SCRIPT": str(CEEVAL / "eval/evaluate_by_GCD.py"),
        "GCD_USE_CUDA": str(GCD_USE_CUDA).lower(),
        "CUDA_VISIBLE_DEVICES": "0",
    })
    print(f"GCD: {model_name}/{content} ({png_count(folder)} ảnh) -> {csv}")
    with log.open("w") as stream:
        subprocess.run(
            ["bash", str(CEEVAL / "run/run_gcd_evaluation_colab.sh"), str(folder), str(xlsx), str(csv)],
            env=env, stdout=stream, stderr=subprocess.STDOUT, check=True,
        )
    print("  hoàn tất; log:", log)

## 7. Tổng hợp kết quả

CE-Eval gốc báo `conditional_accuracy = correct / detected`, tức bỏ các ảnh không tìm thấy mặt. Notebook bổ sung:

- `face_detection_rate = detected / total`
- `identity_hit_rate = correct / total` (không che giấu no-face)

Với split **erase**, các tỷ lệ nhận đúng identity càng thấp càng tốt. Với **retain**, chúng càng cao càng tốt. Nên đọc cả ba cột: một model tạo ảnh không có mặt có thể trông tốt ở erase nhưng rất tệ về khả năng giữ chất lượng/prompt fidelity.

In [ ]:
def summarize_gcd(csv_path, model_name, content):
    df = pd.read_csv(csv_path, index_col=0, keep_default_na=False)
    raw = df["p_celebrity_correct"].astype(str)
    detected = raw.ne("N")
    scores = pd.to_numeric(raw.where(detected), errors="coerce").fillna(0.0)
    correct = detected & scores.gt(0)
    return {
        "model": model_name,
        "split": content,
        "n_images": len(df),
        "n_faces_detected": int(detected.sum()),
        "n_identity_correct": int(correct.sum()),
        "face_detection_rate": float(detected.mean()),
        "conditional_accuracy_CE_Eval": float(correct.sum() / detected.sum()) if detected.any() else 0.0,
        "identity_hit_rate": float(correct.mean()),
        "mean_matched_top1_probability": float(scores.mean()),
    }

rows = []
for model_name, content, _ in eval_targets:
    csv = gcd_output / f"{model_name}_{content}.csv"
    rows.append(summarize_gcd(csv, model_name, content))
summary = pd.DataFrame(rows).sort_values(["split", "model"]).reset_index(drop=True)
display(summary.style.format({
    "face_detection_rate": "{:.3f}",
    "conditional_accuracy_CE_Eval": "{:.3f}",
    "identity_hit_rate": "{:.3f}",
    "mean_matched_top1_probability": "{:.3f}",
}))
summary.to_csv(gcd_output / "summary.csv", index=False)
print("Đã lưu:", gcd_output / "summary.csv")

## 8. Đóng gói kết quả trong persistent volume

Mặc định chỉ nén checkpoint và bảng/log GCD để file vừa phải; ảnh vẫn nằm trong `OUTPUT_ROOT/images`. Dùng Jupyter file browser, SCP hoặc Vast AI file transfer để lấy file zip.

In [ ]:
import shutil
from datetime import datetime

bundle_root = WORKSPACE / f"CEdit-CE-Eval-results-{datetime.now():%Y%m%d-%H%M%S}"
bundle_root.mkdir()
shutil.copytree(OUTPUT_ROOT / "checkpoints", bundle_root / "checkpoints")
shutil.copytree(OUTPUT_ROOT / "gcd", bundle_root / "gcd")
archive = shutil.make_archive(str(bundle_root), "zip", root_dir=bundle_root)
print("Archive created:", archive)
print("Download it through the Vast Jupyter file browser or copy it with SCP.")